In [1]:
import pandas as pd
import numpy as np
import re

# =========================
# 1. CARGA DEL CSV
# =========================

df = pd.read_csv(
    "limpieza_demanda_real_completa_2020_2024.csv",
    sep=";",
    parse_dates=["datetime"]
)

# =========================
# 2. LIMPIEZA DE VALUE
# =========================

#funcion para eliminar los puntos execepto el primero, que indica los decimales
def clean_mixed_number(x):
    if x is None:
        return x
    
    s = str(x).strip()
    
    # Si no tiene puntos o solo hay uno, no hace falta hacer nada
    if s.count(".") <= 1:
        return float(s)
    
    # Dividimos la cadena por los puntos
    parts = s.split(".")
    
    # Unimos todo menos la ultima parte (los deciamles son a partir del primer punto a la derecha)
    integer_part = "".join(parts[:-1])
    decimal_part = parts[-1]
    
    cleaned = integer_part + "." + decimal_part
    
    # Si no tiene punto
    return float(cleaned)


# Aplicar la funcion a la columna value
df["value"] = df["value"].apply(clean_mixed_number)

# Eliminar filas con valores invalidos (na)
#df = df.dropna(subset=["value"])

# =========================
# 3. LIMPIEZA DATETIME
# ==========================

df["datetime"] = df["datetime"].str.replace("T", " ", regex=False)
#df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

# Convertir a datetime si no lo esta ya
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

# Opcional: convertir a hora local España sin timezone
df["datetime"] = df["datetime"].dt.tz_convert("Europe/Madrid")
df["datetime"] = df["datetime"].dt.tz_localize(None)

# =========================
# 3. Eliminar Columna id
# ==========================

df = df.drop('id', axis=1)

# Convertir la columna datetime a datetime y establecerla como índice
df = df.set_index('datetime')

# =========================
# 8. Solucionar Problema del cambio de hora 
# ==========================

#Extraer la hora de la columna datetime y crear una nueva columna "hour" de 0-23
df["hour"] = df.index.hour


# OTOÑO: media de todas las columnas numéricas para las horas duplicadas
df = df.groupby("datetime", as_index=False).mean(numeric_only=True).round(2)

# PRIMAVERA: rellenar horas faltantes
df = df.set_index("datetime").sort_index()

rango = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(rango)

# Interpolar todas las columnas de una vez
df = df.interpolate(method="linear")

df = df.reset_index().rename(columns={"index": "datetime"})

# 9. Ordenar
df = df.sort_values("datetime").reset_index(drop=True)

# Cambiar nombre de una columna
df = df.rename(columns={'value': 'demand'})


print(df.head())

df.to_csv("LimpiezaDemanda2020_2024.csv", index=False, sep=';')


             datetime    demand  hour
0 2020-01-01 00:00:00  23001.00   0.0
1 2020-01-01 01:00:00  22253.67   1.0
2 2020-01-01 02:00:00  20999.50   2.0
3 2020-01-01 03:00:00  19784.17   3.0
4 2020-01-01 04:00:00  18997.33   4.0


C:\Users\Jaime_Sanchez\AppData\Local\Temp\ipykernel_12852\3140468904.py:81: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df = df.groupby("datetime", as_index=False).mean(numeric_only=True).round(2)
